In [17]:
import os
import argparse
import random
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses, models

USER_ID_COL = "user_id"
STREAMER_ID_COL = "streamer_id"

def sample_triplets(train_df, all_streamers):
    """Assumes train_df only contains positive interactions."""
    triplets = [] # (user_id, pos_item_id, neg_item_id)
    for user_id, group in train_df.groupby(USER_ID_COL):
        pos_items = set(group[STREAMER_ID_COL])
        neg_candidates = list(all_streamers - pos_items)
        if not neg_candidates:
            continue

        # Sample one negative for each positive item
        for pos_item in pos_items:
            neg_item = np.random.choice(neg_candidates)
            triplets.append((user_id, pos_item, neg_item))
    return triplets

def build_triplets_from_val_df(val_df):
    """Build (user_id, pos_item_id, neg_item_id) from validation DataFrame with label column."""
    triplets = []
    for user_id, group in val_df.groupby(USER_ID_COL):
        pos_items = group[group["label"] == 1][STREAMER_ID_COL].tolist()
        neg_items = group[group["label"] == 0][STREAMER_ID_COL].tolist()

        if not pos_items or not neg_items:
            continue
        
        pos_item = pos_items[0]  # only one positive per user
        neg_item = random.choice(neg_items)
        
        triplets.append((user_id, pos_item, neg_item))
    
    return triplets

# def build_user_embedding_cache(encoder, user_to_items, item_id_to_text):
#     """Builds a cache of user embeddings by averaging item embeddings."""
#     user_embeddings = {}
#     for user_id, item_ids in tqdm(user_to_items.items(), desc="Building user embeddings"):
#         item_texts = [item_id_to_text[item_id] for item_id in item_ids if item_id in item_id_to_text]
#         if not item_texts:
#             continue
#         user_embedding = encoder.encode(item_texts, convert_to_tensor=True, show_progress_bar=False)
#         user_embedding = torch.mean(user_embedding, dim=0)
#         user_embeddings[user_id] = user_embedding
#     return user_embeddings

def build_user_embedding_cache(encoder, user_to_items, item_id_to_text):
    """Builds a cache of user embeddings by averaging item embeddings."""
    all_texts = []
    all_user_ids = []

    for user_id, item_ids in user_to_items.items():
        for item_id in item_ids:
            if item_id in item_id_to_text:
                all_texts.append(item_id_to_text[item_id]) # item text
                all_user_ids.append(user_id)               # user_id owning the item text

    print(f"› Encoding {len(all_texts)} item texts for {len(user_to_items)} users...")
    all_embeddings = encoder.encode(all_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=128)

    emb_sum = defaultdict(lambda: torch.zeros_like(all_embeddings[0]))
    emb_count = defaultdict(int)

    for uid, emb in zip(all_user_ids, all_embeddings):
        emb_sum[uid] += emb
        emb_count[uid] += 1

    user_embeddings = {uid: emb_sum[uid] / emb_count[uid] for uid in emb_sum}
    return user_embeddings

class TripletDatasetWithUserEmbeddings(Dataset):
    def __init__(self, user_embeddings, item_id_to_text, triplets):
        self.user_embeddings = user_embeddings # {user_id: embedding}
        self.item_id_to_text = item_id_to_text
        self.triplets = triplets
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        user_id, pos_item_id, neg_item_id = self.triplets[idx]

        # get user embedding
        user_emb = self.user_embeddings[user_id]
        pos_item_text = self.item_id_to_text[pos_item_id]
        neg_item_text = self.item_id_to_text[neg_item_id]

        return user_emb, pos_item_text, neg_item_text


class DynamicUserEmbeddingDataset(Dataset):
    def __init__(self, user_to_items, item_id_to_text, triplets, encoder):
        self.user_to_items = user_to_items
        self.item_id_to_text = item_id_to_text
        self.triplets = triplets
        self.encoder = encoder
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        user_id, pos_item_id, neg_item_id = self.triplets[idx]

        # get user's interacted item texts
        user_items = self.user_to_items.get(user_id, [])
        user_item_texts = [self.item_id_to_text[item_id] for item_id in user_items if item_id in self.item_id_to_text]

        # encode user via mean pooling of item embeddings
        user_embedding = self.encoder.encode(user_item_texts, convert_to_tensor=True, show_progress_bar=False)
        user_embedding = torch.mean(user_embedding, dim=0)

        pos_item_text = self.item_id_to_text[pos_item_id]
        neg_item_text = self.item_id_to_text[neg_item_id]

        return InputExample(texts=[user_embedding, pos_item_text, neg_item_text])

def encode_texts(model, texts, device):
    tokens = model.tokenize(texts)  # {'input_ids': ..., 'attention_mask': ...}
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.set_grad_enabled(True):
        outputs = model(tokens)
    return outputs['sentence_embedding']  # [batch_size, hidden_dim]

item_corpus = "/nas02/home/kevin/recsys_pipeline/features/streamer/latest.parquet"
train_interactions = "/nas02/home/kevin/recsys_pipeline/data/splits/donate/interactions_train.parquet"
val_split_path = "/nas02/home/kevin/recsys_pipeline/data/splits/donate/val.parquet"
output_dir = "/nas02/home/kevin/recsys_pipeline/src/embeddings/models/"
epochs = 5
batch_size = 32
margin = 0.2

# load item corpus
item_df = pd.read_parquet(item_corpus) if item_corpus.endswith('.parquet') else pd.read_csv(item_corpus)
item_id_to_text = dict(zip(item_df[STREAMER_ID_COL], item_df["item_sentence"]))

# load training interactions
train_df = pd.read_parquet(train_interactions) if train_interactions.endswith('.parquet') else pd.read_csv(train_interactions)
user_to_items = train_df.groupby(USER_ID_COL)[STREAMER_ID_COL].apply(list).to_dict() # {user_id: [item_ids]}
all_items = set(item_df[STREAMER_ID_COL])

# sample triplets
triplets = sample_triplets(train_df, all_items)
print(f"Sampled {len(triplets)} triplets for training")
triplets = [t for t in triplets if t[0] in user_to_items and t[1] in item_id_to_text and t[2] in item_id_to_text]
print(f"Filtered down to {len(triplets)} valid triplets")

# load validation split
val_df = pd.read_parquet(val_split_path) if val_split_path.endswith('.parquet') else pd.read_csv(val_split_path)
val_triplets = build_triplets_from_val_df(val_df)
print(f"Loaded {len(val_triplets)} validation triplets")

# define shared encoder to be trained
encoder_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
word_model = models.Transformer(encoder_name)
pooling_model = models.Pooling(word_model.get_word_embedding_dimension())
encoder = SentenceTransformer(modules=[word_model, pooling_model])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder.to(device)

# sentence transformers' Triplet loss expects triplets to be all texts
# use `torch.nn.TripletMarginLoss` instead
loss_function = torch.nn.TripletMarginLoss(margin=margin, p=2)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-5)

os.makedirs(output_dir, exist_ok=True)

print("› Training encoder with triplet loss...")
for epoch in range(1, epochs + 1):
    print(f"\n› Epoch {epoch}/{epochs}")

    user_embeddings = build_user_embedding_cache(encoder, user_to_items, item_id_to_text)
    dataset = TripletDatasetWithUserEmbeddings(user_embeddings, item_id_to_text, triplets)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    print(f"Training on {len(dataloader)} batches with batch size {batch_size}")

    total_loss = 0.0
    encoder.train()

    for step, (user_emb, pos_item_text, neg_item_text) in enumerate(tqdm(dataloader, desc="Training")):
        # print(user_emb.shape, len(pos_item_text), len(neg_item_text))
        # move user embedding to device
        user_emb = user_emb.to(device)

        # print(pos_item_text)
        # break

        # encode positive and negative item texts
        pos_emb = encode_texts(encoder, list(pos_item_texts), device)
        neg_emb = encode_texts(encoder, list(neg_item_texts), device)

        loss = loss_function(user_emb, pos_emb, neg_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch} - Average Loss: {avg_loss:.4f}")

    val_loss = evaluate_validation_loss(encoder, loss_function, val_triplets, item_id_to_text, user_to_items, batch_size=batch_size)
    print(f"› Epoch {epoch} validation loss: {val_loss:.4f}")

    # Save model checkpoint
    model_path = os.path.join(output_dir, f"encoder_epoch_{epoch}.pt")
    encoder.save(model_path)
    print(f"Model saved to {model_path}")


print(f"Training complete. Best model saved to {best_model_path}")


Sampled 152109 triplets for training
Filtered down to 152109 valid triplets
Loaded 9661 validation triplets
› Training encoder with triplet loss...

› Epoch 1/5
› Encoding 152109 item texts for 12850 users...


Batches: 100%|██████████| 1189/1189 [00:34<00:00, 34.87it/s]


Training on 4754 batches with batch size 32


Training: 100%|██████████| 4754/4754 [09:20<00:00,  8.49it/s]


Epoch 1 - Average Loss: 0.1056


NameError: name 'evaluate_validation_loss' is not defined

In [14]:
def encode_texts(model, texts, device):
    tokens = model.tokenize(texts)  # {'input_ids': ..., 'attention_mask': ...}
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.set_grad_enabled(True):
        outputs = model(tokens)
    return outputs['sentence_embedding']  # [batch_size, hidden_dim]


pos_emb = encode_texts(encoder, list(pos_item_texts), device)

In [15]:
pos_emb.shape

torch.Size([32, 384])

In [13]:
pos_emb = encoder(list(pos_item_text))["sentence_embedding"]

AttributeError: 'list' object has no attribute 'items'